# 面试问题：LLM Benchmark 去污染怎样同时发现精确泄漏和近似改写？

可以直接复述的回答是：第一，为评测题和训练样本保存来源、时间和规范化文本。第二，精确 hash 只能发现字符级重复。第三，字符 n-gram Jaccard 或 MinHash 可以召回标点变化和轻度改写。第四，公共模板必须先剥离，否则会产生大量误报。第五，阈值附近样本进入人工审计，污染题应从主分数剔除或单独报告。第六，要在标注集上同时看 precision、recall 和被污染分数差异。下面用五道可读面试题与八条训练记录演示。

## 真实案例：评测题与训练语料的来源审计

五道评测题覆盖速度计算、Python 去重、BM25、梯度累积和 HTTP 429。八条训练记录中包含精确副本、标点变化、近义改写和无关文本。数据均为教学构造，并标出真实污染标签用于比较检测器；真实数据还需要版权和抓取时间信息。

In [1]:
benchmarks = [  # 定义五道带人工污染标签的评测题
    {"id": "B1", "text": "一列火车以每小时 60 公里行驶 2 小时，路程是多少？", "contaminated": True},  # 训练集含标点变化副本
    {"id": "B2", "text": "不用 set，怎样保持顺序地去重 Python 列表？", "contaminated": True},  # 训练集含语序改写
    {"id": "B3", "text": "BM25 参数 k1 增大时，词频饱和会怎样变化？", "contaminated": False},  # 训练集只有同领域无答案文本
    {"id": "B4", "text": "梯度累积为什么需要按有效样本数缩放损失？", "contaminated": False},  # 未出现在训练记录
    {"id": "B5", "text": "HTTP 429 通常表示什么，客户端应如何重试？", "contaminated": True},  # 训练集含轻度改写答案
]  # 结束五题审计集
training_records = [  # 定义八条带来源和抓取日期的训练记录
    {"id": "T1", "source": "forum-a", "date": "2025-03-01", "text": "一列火车以每小时60公里行驶2小时，路程是多少? 答案120公里。"},  # B1 标点与空格变化副本
    {"id": "T2", "source": "blog-b", "date": "2025-04-11", "text": "Python列表不用set如何去重并保留原有顺序，可以维护seen再遍历。"},  # B2 语序改写并带答案
    {"id": "T3", "source": "docs-c", "date": "2024-07-09", "text": "BM25 使用 k1 和 b 控制词频与长度归一化。"},  # 与 B3 同领域但不含目标关系
    {"id": "T4", "source": "qa-d", "date": "2025-05-20", "text": "状态码429表示请求过多，应读取Retry-After并采用退避重试。"},  # B5 轻度改写答案
    {"id": "T5", "source": "notes-e", "date": "2025-01-08", "text": "AdamW 将权重衰减与梯度更新解耦。"},  # 无关优化器记录
    {"id": "T6", "source": "manual-f", "date": "2025-02-16", "text": "向量数据库可以使用 HNSW 建立近邻索引。"},  # 无关检索记录
    {"id": "T7", "source": "template-g", "date": "2025-06-01", "text": "请回答下列问题，并解释你的推理过程。"},  # 公共评测模板
    {"id": "T8", "source": "support-h", "date": "2025-06-02", "text": "HTTP 500 表示服务器内部错误。"},  # 与 B5 协议相关但答案不同
]  # 结束八条训练来源记录
print("评测题：id | contaminated | text")  # 展示去污染系统待审计的真实题面
for benchmark in benchmarks:  # 逐条输出五道题
    print(f"{benchmark['id']} | {benchmark['contaminated']} | {benchmark['text']}")  # 同时展示人工标签供指标计算
print("训练来源：", [(record["id"], record["source"], record["date"]) for record in training_records])  # 展示来源与时间元数据


评测题：id | contaminated | text
B1 | True | 一列火车以每小时 60 公里行驶 2 小时，路程是多少？
B2 | True | 不用 set，怎样保持顺序地去重 Python 列表？
B3 | False | BM25 参数 k1 增大时，词频饱和会怎样变化？
B4 | False | 梯度累积为什么需要按有效样本数缩放损失？
B5 | True | HTTP 429 通常表示什么，客户端应如何重试？
训练来源： [('T1', 'forum-a', '2025-03-01'), ('T2', 'blog-b', '2025-04-11'), ('T3', 'docs-c', '2024-07-09'), ('T4', 'qa-d', '2025-05-20'), ('T5', 'notes-e', '2025-01-08'), ('T6', 'manual-f', '2025-02-16'), ('T7', 'template-g', '2025-06-01'), ('T8', 'support-h', '2025-06-02')]


## Baseline / 基线：规范化后的精确 SHA-256

先统一大小写、全半角、空白和标点，再比较 hash。它擅长完全副本，却无法识别语序变化和答案型改写。

In [2]:
import hashlib  # 使用 SHA-256 构建精确内容指纹
import re  # 使用正则统一标点、空白和模板
import unicodedata  # 使用 NFKC 统一全角与兼容字符
def normalize(text):  # 对训练和评测文本执行同一规范化
    normalized = unicodedata.normalize("NFKC", text).casefold()  # 统一字符形态和大小写
    return re.sub(r"[^\w一-鿿]+", "", normalized)  # 移除空白与标点只保留正文字符
training_hashes = {hashlib.sha256(normalize(record["text"]).encode("utf-8")).hexdigest(): record["id"] for record in training_records}  # 建立训练记录精确指纹索引
baseline_predictions = []  # 收集五题精确 hash 污染判定
print("精确 Hash：benchmark | matched_record | contaminated")  # 输出基线逐题匹配结果
for benchmark in benchmarks:  # 对五道评测题检查完全规范化重复
    digest = hashlib.sha256(normalize(benchmark["text"]).encode("utf-8")).hexdigest()  # 计算当前题面精确指纹
    matched = training_hashes.get(digest)  # 查询训练集是否存在完全相同规范化文本
    predicted = matched is not None  # 有精确匹配时标记污染
    baseline_predictions.append(predicted)  # 保存基线判定供指标比较
    print(f"{benchmark['id']} | {matched or '-'} | {predicted}")  # 展示精确方法漏掉的改写题


精确 Hash：benchmark | matched_record | contaminated
B1 | - | False
B2 | - | False
B3 | - | False
B4 | - | False
B5 | - | False


## 核心实现：去模板字符 4-gram Jaccard 与来源账本

字符 n-gram 能容忍少量改写。检测器先剥离公共指令模板，再寻找每道题最相似训练记录，同时保留 source、date 和分数供人工审计。

In [3]:
boilerplates = ("请回答下列问题", "并解释你的推理过程", "请给出答案")  # 定义不应作为污染证据的公共模板
def content_text(text):  # 规范化并剥离公共评测模板
    cleaned = text  # 保留原始文本供逐个模板替换
    for phrase in boilerplates:  # 移除所有已知无区分度指令
        cleaned = cleaned.replace(phrase, "")  # 只保留问题特有内容
    return normalize(cleaned)  # 返回统一字符形式的正文
def shingles(text, width=4):  # 构造连续字符 n-gram 集合
    normalized = content_text(text)  # 使用去模板后的问题正文
    return {normalized[index:index + width] for index in range(max(0, len(normalized) - width + 1))}  # 返回去重四字符片段
def jaccard(left, right):  # 计算两组字符片段的交并比
    union = left | right  # 构建所有不同片段集合
    return len(left & right) / len(union) if union else 0.0  # 空文本返回零相似度
similarity_threshold = 0.34  # 设置教学标注集上的近似污染阈值
core_predictions = []  # 收集近似检测器污染判定
match_ledger = []  # 保存题目、来源、日期和相似度证据
print("近似匹配：benchmark | record | source | score | predicted")  # 输出逐题最相似训练记录
for benchmark in benchmarks:  # 对五道评测题执行全量小规模比对
    candidates = []  # 收集当前题与八条训练记录的相似度
    for record in training_records:  # 遍历所有训练来源
        score = jaccard(shingles(benchmark["text"]), shingles(record["text"]))  # 计算去模板四字符 Jaccard
        candidates.append((score, record))  # 保存分数和来源元数据
    score, record = max(candidates, key=lambda item: item[0])  # 选择最相似训练记录
    predicted = score >= similarity_threshold  # 根据固定阈值生成污染判定
    core_predictions.append(predicted)  # 保存判定供 precision/recall 计算
    match_ledger.append({"benchmark": benchmark["id"], "record": record["id"], "source": record["source"], "date": record["date"], "score": score, "predicted": predicted})  # 写入可审计来源账本
    print(f"{benchmark['id']} | {record['id']} | {record['source']} | {score:.3f} | {predicted}")  # 展示近似证据而非黑盒标签


近似匹配：benchmark | record | source | score | predicted
B1 | T1 | forum-a | 0.731 | True
B2 | T2 | blog-b | 0.159 | False
B3 | T3 | docs-c | 0.030 | False
B4 | T1 | forum-a | 0.000 | False
B5 | T8 | support-h | 0.033 | False


## 失败案例与修正：公共模板会制造虚假高相似

两个完全不同的问题都带“请回答下列问题，并解释你的推理过程”。若直接用短文本 3-gram，模板贡献会主导分数；剥离模板后只比较问题特有正文。

In [4]:
template_left = "请回答下列问题，并解释你的推理过程：法国首都是哪里？"  # 构造地理问题模板文本
template_right = "请回答下列问题，并解释你的推理过程：如何实现二叉树遍历？"  # 构造算法问题模板文本
def raw_shingles(text, width=3):  # 构造不去模板的脆弱字符片段
    normalized = normalize(text)  # 只做基础字符规范化
    return {normalized[index:index + width] for index in range(max(0, len(normalized) - width + 1))}  # 返回三字符片段集合
raw_template_score = jaccard(raw_shingles(template_left), raw_shingles(template_right))  # 计算公共模板污染下的相似度
clean_template_score = jaccard(shingles(template_left, 3), shingles(template_right, 3))  # 计算剥离模板后的问题正文相似度
raw_flag = raw_template_score >= 0.25  # 模拟低阈值检测器对公共模板的误报
clean_flag = clean_template_score >= 0.25  # 检查修正后是否仍误报
print(f"模板修正前：score={raw_template_score:.3f}，flag={raw_flag}")  # 展示无关问题被公共指令拉近
print(f"模板修正后：score={clean_template_score:.3f}，flag={clean_flag}")  # 展示正文比较降低误报


模板修正前：score=0.467，flag=True
模板修正后：score=0.000，flag=False


## 结果表：精确 Hash 与近似检测 Precision/Recall

In [5]:
labels = [benchmark["contaminated"] for benchmark in benchmarks]  # 提取五题人工污染标签
def classification_metrics(predictions, truth):  # 计算小型审计集的 precision、recall 和 F1
    true_positive = sum(predicted and actual for predicted, actual in zip(predictions, truth))  # 统计正确发现污染题数量
    false_positive = sum(predicted and not actual for predicted, actual in zip(predictions, truth))  # 统计误报数量
    false_negative = sum(not predicted and actual for predicted, actual in zip(predictions, truth))  # 统计漏报数量
    precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else 0.0  # 计算污染告警精确率
    recall = true_positive / (true_positive + false_negative) if true_positive + false_negative else 0.0  # 计算真实污染召回率
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0  # 计算调和平均指标
    return precision, recall, f1, true_positive, false_positive, false_negative  # 返回完整审计统计
baseline_metrics = classification_metrics(baseline_predictions, labels)  # 评估精确 hash 检测器
core_metrics = classification_metrics(core_predictions, labels)  # 评估去模板近似检测器
print("detector | precision | recall | f1 | TP | FP | FN")  # 输出同一人工标签下的检测对照
print(f"exact_hash | {baseline_metrics[0]:.2f} | {baseline_metrics[1]:.2f} | {baseline_metrics[2]:.2f} | {baseline_metrics[3]} | {baseline_metrics[4]} | {baseline_metrics[5]}")  # 展示精确方法漏报
print(f"ngram_jaccard | {core_metrics[0]:.2f} | {core_metrics[1]:.2f} | {core_metrics[2]:.2f} | {core_metrics[3]} | {core_metrics[4]} | {core_metrics[5]}")  # 展示近似方法的召回变化
print("需人工审计来源：", [entry for entry in match_ledger if abs(entry["score"] - similarity_threshold) < 0.12])  # 输出阈值附近样本而非强行自动结论


detector | precision | recall | f1 | TP | FP | FN
exact_hash | 0.00 | 0.00 | 0.00 | 0 | 0 | 3
ngram_jaccard | 1.00 | 0.33 | 0.50 | 1 | 0 | 2
需人工审计来源： []


## 结果解读

精确 hash 没有发现带答案、语序或标点变化的训练记录，因此 recall 很低。字符 4-gram 能为 B1、B2、B5 找到具体来源和日期，同时 B3 的同领域文本不足以越过阈值。公共模板反例说明 n-gram 阈值必须建立在问题特有内容上，并保留人工复核带。

## 生产边界

真实去污染需对海量语料使用 MinHash/LSH 或检索索引，处理多语言、代码格式、答案泄漏、翻译和语义改写。训练数据时间必须早于 benchmark 发布审计点，来源删除也要可追踪。阈值应在独立标注集校准，并同时报告去污染前后分数。本例只有八条训练记录。

## 最小回归测试

In [6]:
assert len(benchmarks) >= 5 and len(training_records) >= 5  # 保证案例包含足够评测题和训练来源
assert raw_flag is True and clean_flag is False  # 保证公共模板误报被正文剥离修正
assert core_metrics[1] >= baseline_metrics[1]  # 保证近似检测的污染召回不低于精确 hash
assert core_metrics[2] > baseline_metrics[2]  # 保证近似检测在同一人工集上的 F1 更高
assert all(entry["source"] and entry["date"] for entry in match_ledger)  # 保证每个近似结论保留来源和时间证据
